In [1]:
import pandas as pd 
import numpy as np
import sqlite3

In [5]:
print(sqlite3.sqlite_version)

3.45.3


In [2]:
data = pd.read_csv('../data/data_sessions.csv', parse_dates=['datetime'])
data.head(5)

,EventName,DeviceIDHash,ExpId,datetime,date,session
0,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06,1
1,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06,1
2,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1
3,CartScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1
4,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241298 entries, 0 to 241297
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   EventName     241298 non-null  object        
 1   DeviceIDHash  241298 non-null  int64         
 2   ExpId         241298 non-null  int64         
 3   datetime      241298 non-null  datetime64[ns]
 4   date          241298 non-null  object        
 5   session       241298 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 11.0+ MB


Ранее рассматривал действие клиента внутри одной сессии. Но, очень много сессий, где есть только покупка, но нет карзнины или акций перед покупкой. 
Может клиент делает покупки не в одну сессию. Клиент может откладыать товары в корзину, а потом в другую сессию заходить и покупать их.
Как клиенты доходят до покупки. Посмотреть на процессе не со стороны сессий, а последовательности действий между покупками
То есть факт покупки (или факт первого захода в ПО) - какие-то действия - покупка - какие-то действия - покупка.... не привязываясь к сессии.

In [ ]:
копия от которой отталкиваюсь. буду переделывать ее, так как ARRAY_AGG и STRING_AGG не работает в sqllite!!

conn = sqlite3.connect(':memory:')

data.to_sql('events', conn, index = False, if_exists='replace')

# уберем шум (действия, которые повторялись друг за другом)
# далее разобьем все на группы между покупками, чтобы выяснить, какой путь проходит пользователь от покупки к покупке
query = """ 
-- убираем последовательные дубли, фиксируем начало каждой сессии (нужно для join когда нужно будет пределить начало пути до покупки)
WITH prev_events AS (
SELECT  DeviceIDHash, 
        EventName,
        session,
        datetime,
        MIN(datetime) OVER(PARTITION BY DeviceIDHash, session ORDER BY datetime) start_session,
        LAG(EventName) OVER(PARTITION BY DeviceIDHash, session ORDER BY datetime) prev_event   
FROM events 
),
without_duble AS(
SELECT  DeviceIDHash,
        EventName,
        session,
        datetime,
        start_session
  FROM with_prev
  WHERE prev_event IS NULL OR EventName != prev_event
),
-- упорядочиваем все покупки каждого пользователя с фиксацией времени предыдущей покупки
pay_event AS (
SELECT  DeviceIDHash, 
        session,
        datetime AS paytime,
        ROW_NUMBER() OVER(PARTITION BY DeviceIDHash ORDER BY datetime) AS number_pay,
        LAG(datetime) OVER(PARTITION BY DeviceIDHash ORDER BY datetime) AS prev_paytime
FROM without_duble
WHERE EventName = 'PaymentScreenSuccessful'
),
-- объединяем таблицы. Каждой покупке мы сопоставляем события, которые шли до нее и были после предыдущей покупки. 
-- Если предыдущей покупки не было, то от начала сессии (первой активности в логе)
active_pay AS (
SELECT  pe.DeviceIDHash, 
        pe.number_pay,
        pe.session AS pay_session,
        pe.paytime,
        pe.prev_paytime,
        wd.EventName,
        wd.session AS active_session,
        wd.datetime AS active_time,
        wd.number_active
FROM  pay_event pe
JOIN without_duble wd
ON  pe.DeviceIDHash = wd.DeviceIDHash AND
    wd.datetime < pe.paytime AND
    wd.datetime >= COALESCE(pe.prev_paytime, wd.start_session) AND
    wd.EventName != 'PaymentScreenSuccessful' 
),
-- Убираем повторяющиеся действия с событиях, сопутствующих покупке. Из дублей оставляем только первое вхождение
pay_path AS(
SELECT  DeviceIDHash, 
        number_pay,
        pay_session,
        paytime,
        EventName,
        MIN(active_time) active_time,
        MIN(active_session) active_session
        --ROW_NUMBER() OVER(PARTITION BY DeviceIDHash, number_pay) number_active
FROM active_pay
GROUP BY DeviceIDHash, number_pay, pay_session, paytime, EventName
)
-- схлопываем все, чтобы проанализировать пути. сейчас через ARRAY_AGG с учетом времени, то есть фокусируемся именно на последовательности действий
SELECT  DeviceIDHash, 
        number_pay,
        pay_session,
        paytime,
        ARRAY_AGG(EventName ORDER BY active_time) step_active, (или использовать STRING_AGG)
        ARRAY_AGG(active_session ORDER BY active_time) step_time,
        COUNT(EventName) sum_steps,
        MAX(CASE WHEN event = 'CartScreenAppear' THEN 1 ELSE 0 END) as cart_exist,
        MAX(CASE WHEN event = 'OffersScreenAppear' THEN 1 ELSE 0 END) as offers_exist,
        MAX(CASE WHEN event = 'MainScreenAppear' THEN 1 ELSE 0 END) as main_exist
FROM pay_path
GROUP BY DeviceIDHash, number_pay, pay_session, paytime

"""
result = pd.read_sql_query(query, conn)
conn.close()

In [99]:
conn = sqlite3.connect(':memory:')

data.to_sql('events', conn, index = False, if_exists='replace')

# уберем шум (действия, которые повторялись друг за другом)
# далее разобьем все на группы между покупками, чтобы выяснить, какой путь проходит пользователь от покупки к покупке
query = """ 
-- убираем последовательные дубли, фиксируем начало каждой сессии (нужно для join когда нужно будет пределить начало пути до покупки)
WITH prev_events AS (
SELECT  DeviceIDHash, 
        EventName,
        session,
        datetime,
        MIN(datetime) OVER(PARTITION BY DeviceIDHash, session ORDER BY datetime) start_session,
        LAG(EventName) OVER(PARTITION BY DeviceIDHash, session ORDER BY datetime) prev_event   
FROM events 
),
without_duble AS(
SELECT  DeviceIDHash,
        EventName,
        session,
        datetime,
        start_session
FROM prev_events
WHERE prev_event IS NULL OR EventName != prev_event
),
-- упорядочиваем все покупки каждого пользователя с фиксацией времени предыдущей покупки. Покупки одной сессии считаем как за одну, но фиксируем их количество
session_pay AS (
SELECT  DeviceIDHash, 
        session,
        MAX(datetime) AS paytime,
        COUNT(*) AS cnt_pay_in_session
        --ROW_NUMBER() OVER(PARTITION BY DeviceIDHash,session ORDER BY datetime) AS number_pay, -- добавил в партишн session, чтобы считать продажи в одной сессии как за одну
        --LAG(datetime) OVER(PARTITION BY DeviceIDHash,session ORDER BY datetime) AS prev_paytime оконка с группировкой не работают, разбил на 2 СТЕ
FROM without_duble
WHERE EventName = 'PaymentScreenSuccessful'
GROUP BY DeviceIDHash,session
),
all_pay_of_user AS (
SELECT  DeviceIDHash, 
        session,
        paytime,
        cnt_pay_in_session, 
        ROW_NUMBER() OVER(PARTITION BY DeviceIDHash ORDER BY paytime) AS number_pay, 
        LAG(paytime) OVER(PARTITION BY DeviceIDHash ORDER BY paytime) AS prev_paytime
FROM session_pay
),
-- объединяем таблицы. Каждой покупке мы сопоставляем события, которые шли до нее и были после предыдущей покупки. 
-- Если предыдущей покупки не было, то от начала сессии (первой активности в логе)
active_before_pay AS (
SELECT  pou.DeviceIDHash, 
        pou.number_pay,
        pou.session AS pay_session,
        pou.cnt_pay_in_session,
        pou.paytime,
        pou.prev_paytime,
        wd.EventName,
        wd.session AS active_session,
        wd.datetime AS active_time
FROM  all_pay_of_user pou
JOIN without_duble wd
ON  pou.DeviceIDHash = wd.DeviceIDHash AND
    wd.datetime <= pou.paytime AND
    wd.datetime >= COALESCE(pou.prev_paytime, wd.start_session) --AND
    --wd.EventName != 'PaymentScreenSuccessful' 
),
-- Убираем повторяющиеся действия с событиях, сопутствующих покупке. Из дублей оставляем только последние (ранее были первые) вхождения
path_to_pay AS (
SELECT  DeviceIDHash, 
        number_pay,
        pay_session,
        cnt_pay_in_session,
        paytime,
        EventName,
        MAX(active_time) active_time,
        MAX(active_session) active_session
FROM active_before_pay
GROUP BY DeviceIDHash, number_pay, pay_session, cnt_pay_in_session, paytime, EventName
)
-- схлопываем все, чтобы проанализировать пути. сейчас через ARRAY_AGG с учетом времени, то есть фокусируемся именно на последовательности действий
SELECT  DeviceIDHash, 
        number_pay,
        pay_session,
        cnt_pay_in_session,
        paytime,
        GROUP_CONCAT(EventName, '-' ORDER BY active_time) step_active, 
        GROUP_CONCAT(active_session, '-' ORDER BY active_time) step_session,
        COUNT(EventName) sum_steps,
        MAX(CASE WHEN EventName = 'CartScreenAppear' THEN 1 ELSE 0 END) as cart_exist,
        MAX(CASE WHEN EventName = 'OffersScreenAppear' THEN 1 ELSE 0 END) as offers_exist,
        MAX(CASE WHEN EventName = 'MainScreenAppear' THEN 1 ELSE 0 END) as main_exist
FROM path_to_pay
GROUP BY DeviceIDHash, number_pay, pay_session, cnt_pay_in_session, paytime

"""
result = pd.read_sql_query(query, conn, parse_dates=['paytime'])
conn.close()

In [92]:
len(result)

12555

In [48]:
print(result.to_string(max_colwidth=None))

              DeviceIDHash  number_pay  pay_session  cnt_pay_in_session              paytime                                                                            step_active      step_time  sum_steps  cart_exist  offers_exist  main_exist
0         6909561520679493           1            1                   1  2019-08-06 18:52:58                              CartScreenAppear-MainScreenAppear-PaymentScreenSuccessful          1-1-1          3           1             0           1
1         6922444491712477           1            1                   1  2019-08-04 14:19:40                              CartScreenAppear-MainScreenAppear-PaymentScreenSuccessful          1-1-1          3           1             0           1
2         6922444491712477           2            2                   1  2019-08-04 17:16:32           CartScreenAppear-OffersScreenAppear-MainScreenAppear-PaymentScreenSuccessful        1-1-2-2          4           1             1           1
3         69224444917124

In [102]:
# так как наблюдается хаотичность в действиях из-за отсутствия строкой последовательности в событиях, то будем рассматривать не на последовательность событий, а на ФАКТ присутствия события в цепочке
# 
result['step_active'] = result['step_active'].apply(lambda x: '-'.join(sorted(x.split('-')[0:-1]))) #result['step_active'].apply(lambda x: x.split('-')[0:-1]).apply(lambda lst: lst.sort() or lst)

In [105]:
# у некоторых пользователей в логе первым действием является покупка (всего таких случаев 51, 27 из них в первый день логирования). 
# Скорее всего это те, кто делал предварительные действия в более ранниюе даты(зашел в ПО, положил товар в корзину, просмотрел акции и т.п), 
#и поэтому  не попали к нам в лог. Возможно, была рассылка с акциями и это прямая продажа от диплинка. 
# Удалять ли такие строки - в реалии спросить про рассылки или взять более полный лог и отследить именно этих пользователй.
# Так как такой возможности нет, буду считать, что это "неполные продажи" и удалю данные по таким продажам за первые два дня наблюдаемого периода
result = result[~((result['step_active'] == '') & (result['number_pay'] == 1) & (result['paytime'] < pd.to_datetime('2019-08-03')))]

In [128]:
# воспользуемся SQL еще раз и сгруппируем данные по ФАКТАМ присутвия в процессе покупки таких действий, как MainScreenAppear, OffersScreenAppear, CartScreenAppear

conn = sqlite3.connect(':memory:')
result.to_sql('path_to_pay', conn, index=False, if_exists='replace')

query = """ 
WITH total_pay AS(
SELECT COUNT(*) total FROM path_to_pay
)
SELECT  main_exist AS 'Запускал приложение',
        offers_exist AS 'Просматривал акции',
        cart_exist AS 'Заходил в корзину',
        COUNT(*) AS 'Зафиксировано фактов',
        ROUND(100 * COUNT(*)*1.0 / (SELECT total FROM total_pay),2) AS 'Доля, %',
        ROUND(AVG(cnt_pay_in_session),2)  AS 'Ср.коли-во покупок за сессию'
FROM path_to_pay
GROUP BY main_exist, offers_exist, cart_exist
--ORDER BY 4


"""
svod = pd.read_sql_query(query, conn)

In [141]:
svod.to_clipboard(index = False, sep = ' ')

In [130]:
# проверка через датафрейм
print(result.groupby(['main_exist','cart_exist',  'offers_exist']).agg(count = ('cart_exist', 'count'), avg_cnt_pay = ('cnt_pay_in_session', 'mean')).reset_index())

   main_exist  cart_exist  offers_exist  count  avg_cnt_pay
0           0           0             0     20     1.000000
1           0           0             1     59     1.000000
2           0           1             0    285     1.470175
3           0           1             1    928     3.262931
4           1           0             0    216     1.004630
5           1           0             1    319     1.006270
6           1           1             0   1308     1.104740
7           1           1             1   9383     2.717574


In [136]:
svod[svod['Запускал приложение'] == 0]['Зафиксировано фактов'].sum()/svod['Зафиксировано фактов'].sum() # доля покупок через дипклики, пуши, прямая ссылка оплаты... 

np.float64(0.10321137561910848)

In [ ]:
svod